# Manhua Lens — PRIVATE Korean voice preparation

OpenVoice V2 + MeloTTS Korean. Free Kaggle GPU/CPU session, or free Colab.
Keep this notebook PRIVATE, enable Internet, choose a free GPU when available,
and turn off session persistence. No paid service or tunnel is used.

**Run cells interactively in order; do not use Save Version / Run All after uploading.**
The reference WAV is uploaded using a session widget, never a Kaggle Dataset.
Raw audio and intermediate chunks live under `/tmp`, outside `/kaggle/working`.
Your embedding, cloned samples, and downloaded ZIP are also PRIVATE. Never
publish notebook outputs, create a dataset from them, or commit them to GitHub.
The download cell deliberately embeds the private ZIP in its output: clear all
outputs before saving/exporting the notebook, then end the session.

Kaggle is used for preparation, not as a dependable live endpoint. The smallest
local mode plays only exact prepared phrases; all other phrases use device TTS.
Optional arbitrary-text cloning still requires local MeloTTS + OpenVoice models.


In [ ]:
# Embedded PUBLIC helper source: works before repository changes are pushed.
import os
import sys
import subprocess
from pathlib import Path
SCRIPTS = Path('/tmp/manhua-lens/voice_server')
SCRIPTS.mkdir(parents=True, exist_ok=True)
PUBLIC_SOURCES = {'voice_assets.py': '"""Private phrase-cache format shared by preparation and playback."""\nimport hashlib\nimport io\nimport json\nimport re\nimport unicodedata\nimport wave\nimport zipfile\nfrom pathlib import Path\n\nMAX_WAV = 32 * 1024 * 1024\n\n\ndef normalize_text(text):\n    if not isinstance(text, str):\n        raise ValueError("text must be a string")\n    text = unicodedata.normalize("NFC", text).strip()\n    if not text or len(text) > 400:\n        raise ValueError("text must contain 1–400 characters")\n    return text\n\n\ndef cache_name(text):\n    return hashlib.sha256(normalize_text(text).encode("utf-8")).hexdigest() + ".wav"\n\n\ndef validate_wav(data):\n    if len(data) > MAX_WAV:\n        raise ValueError("WAV is too large")\n    try:\n        with wave.open(io.BytesIO(data), "rb") as audio:\n            if audio.getnframes() == 0 or audio.getnchannels() not in (1, 2):\n                raise ValueError("Empty or unsupported WAV")\n            expected = audio.getnframes() * audio.getnchannels() * audio.getsampwidth()\n            if len(audio.readframes(audio.getnframes())) != expected:\n                raise ValueError("Truncated WAV")\n    except (wave.Error, EOFError) as exc:\n        raise ValueError("Expected a PCM WAV") from exc\n    return data\n\n\ndef read_cached(root, text):\n    path = Path(root) / "cache" / cache_name(text)\n    if not path.is_file():\n        return None\n    if path.stat().st_size > MAX_WAV:\n        raise ValueError("WAV is too large")\n    return validate_wav(path.read_bytes())\n\n\ndef import_bundle(archive, destination):\n    """Import only explicit asset names; never extract arbitrary archive paths."""\n    destination = Path(destination)\n    with zipfile.ZipFile(archive) as bundle:\n        infos = bundle.infolist()\n        if len(infos) > 10002 or sum(i.file_size for i in infos) > 512 * 1024 * 1024:\n            raise ValueError("Bundle exceeds the 512 MiB / 10,000 phrase limit")\n        names = [i.filename for i in infos]\n        if len(names) != len(set(names)) or "manifest.json" not in names:\n            raise ValueError("Missing manifest or duplicate bundle entries")\n        for info in infos:\n            name = info.filename\n            if name not in ("manifest.json", "target_se.pth") and not re.fullmatch(r"cache/[0-9a-f]{64}\\.wav", name):\n                raise ValueError("Unexpected bundle entry: " + name)\n            if info.file_size > (MAX_WAV if name.endswith(".wav") else 1024 * 1024):\n                raise ValueError("Oversized bundle entry: " + name)\n        manifest = json.loads(bundle.read("manifest.json"))\n        if not isinstance(manifest, dict) or manifest.get("format") != "manhua-lens-private-v1":\n            raise ValueError("Unsupported private bundle format")\n        for name in names:\n            if name.endswith(".wav"):\n                validate_wav(bundle.read(name))\n        destination.mkdir(parents=True, exist_ok=True)\n        for name in names:\n            target = destination / name\n            if destination.is_symlink() or target.is_symlink() or target.parent.is_symlink():\n                raise ValueError("Refusing a symlink in the private asset directory")\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(bundle.read(name))\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser(description="Import your own PRIVATE Kaggle bundle")\n    parser.add_argument("archive", type=Path)\n    parser.add_argument("destination", type=Path)\n    args = parser.parse_args()\n    import_bundle(args.archive, args.destination)\n    print("Private assets imported. Do not commit or share this directory.")\n', 'engine.py': '"""Optional full OpenVoice V2 + MeloTTS Korean inference."""\nimport tempfile\nimport threading\nfrom pathlib import Path\n\n\nclass KoreanVoice:\n    def __init__(self, checkpoints, embedding, device=None):\n        import torch\n        from melo.api import TTS\n        from openvoice.api import ToneColorConverter\n        from melo.text import korean\n        from korean_frontend import create_phonemizer\n\n        korean.g2p_kr = create_phonemizer()\n\n        self.lock = threading.Lock()\n        self.device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")\n        root = Path(checkpoints)\n        self.converter = ToneColorConverter(str(root / "converter/config.json"), device=self.device)\n        self.converter.load_ckpt(str(root / "converter/checkpoint.pth"))\n        self.target = torch.load(embedding, map_location=self.device, weights_only=True)\n        if not isinstance(self.target, torch.Tensor) or self.target.ndim != 3 or not torch.isfinite(self.target).all():\n            raise ValueError("Invalid target speaker embedding")\n        self.model = TTS(language="KR", device=self.device)\n        speakers = self.model.hps.data.spk2id\n        if "KR" not in speakers:\n            raise RuntimeError("Expected the MeloTTS Korean KR speaker")\n        self.speaker = speakers["KR"]\n        self.source = torch.load(root / "base_speakers/ses/kr.pth", map_location=self.device, weights_only=True)\n        if self.target.shape != self.source.shape:\n            raise ValueError("Target embedding does not match OpenVoice V2")\n\n    def synthesize(self, text):\n        if not self.lock.acquire(blocking=False):\n            raise RuntimeError("Voice service is busy")\n        try:\n            with tempfile.TemporaryDirectory(prefix="mhl-tts-") as tmp:\n                source, output = Path(tmp) / "source.wav", Path(tmp) / "cloned.wav"\n                self.model.tts_to_file(text, self.speaker, str(source), speed=0.95, quiet=True)\n                self.converter.convert(audio_src_path=str(source), src_se=self.source,\n                                       tgt_se=self.target, output_path=str(output), message="@ManhuaLens")\n                return output.read_bytes()\n        finally:\n            self.lock.release()\n', 'korean_frontend.py': '"""Use prebuilt python-mecab-ko on Windows as well as Linux."""\n\n\ndef create_phonemizer():\n    from g2pkk import G2p\n    from mecab import MeCab\n\n    class PortableG2p(G2p):\n        def check_mecab(self):\n            # Dependencies are installed explicitly during setup. Never invoke\n            # g2pkk\'s implicit Windows `pip install eunjeon` / compiler path.\n            pass\n\n        def get_mecab(self):\n            return MeCab()\n\n    return PortableG2p()\n', 'setup_runtime.py': '"""Reproducible model setup inside a Python 3.10 venv (Windows or notebook)."""\nimport argparse\nimport json\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nOPENVOICE_REV = "74a1d147b17a8c3092dd5430504bd83ef6c7eb23"\nMELO_REV = "209145371cff8fc3bd60d7be902ea69cbdb7965a"\nCHECKPOINT_REV = "f36e7edfe1684461a8343844af60babc2efbb727"\nHERE = Path(__file__).resolve().parent\n\n\ndef run(*args):\n    subprocess.run([str(a) for a in args], check=True)\n\n\ndef download_checkpoints(destination):\n    from huggingface_hub import hf_hub_download\n\n    destination = Path(destination)\n    # Only the converter and Korean base speaker are needed for this pipeline.\n    files = ["converter/config.json", "converter/checkpoint.pth", "base_speakers/ses/kr.pth"]\n    if all((destination / name).is_file() for name in files):\n        return\n    for name in files:\n        cached = hf_hub_download("myshell-ai/OpenVoiceV2", name, revision=CHECKPOINT_REV)\n        target = destination / name\n        target.parent.mkdir(parents=True, exist_ok=True)\n        partial = target.with_suffix(target.suffix + ".partial")\n        shutil.copyfile(cached, partial)\n        partial.replace(target)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--torch-index", choices=["cpu", "cu121"], default="cpu")\n    parser.add_argument("--checkpoints", type=Path, default=HERE / "OpenVoice/checkpoints_v2")\n    args = parser.parse_args()\n    if sys.version_info[:2] != (3, 10) or sys.prefix == sys.base_prefix:\n        raise SystemExit("Run this with Python 3.10 inside an isolated venv. See README.md.")\n    pip = [sys.executable, "-m", "pip"]\n    run(*pip, "install", "pip==24.3.1", "setuptools==69.5.1", "wheel==0.45.1")\n    run(*pip, "install", "torch==2.5.1", "torchaudio==2.5.1", "--index-url",\n        "https://download.pytorch.org/whl/" + args.torch_index)\n    run(*pip, "install", "-r", HERE / "requirements-models.txt", "-r", HERE / "requirements.txt")\n    # Install audited inference dependencies explicitly, avoiding old OpenVoice ASR\n    # pins and unused Gradio servers. No faster-whisper/Whisper extraction is used.\n    for repo, revision in [("OpenVoice", OPENVOICE_REV), ("MeloTTS", MELO_REV)]:\n        run(*pip, "install", "--no-deps", "--no-build-isolation",\n            f"https://github.com/myshell-ai/{repo}/archive/{revision}.zip")\n    run(sys.executable, "-m", "nltk.downloader", "-d", Path(sys.prefix) / "nltk_data",\n        "cmudict", "averaged_perceptron_tagger", "punkt")\n    # Melo\'s eager Japanese cleaner imports require the dictionary even for KR.\n    run(sys.executable, "-m", "unidic", "download")\n    run(sys.executable, "-c", "import sys; sys.path.insert(0, " + repr(str(HERE)) + "); from korean_frontend import create_phonemizer; assert create_phonemizer()(\'안녕하세요\'); print(\'Korean phonemizer OK\')")\n    download_checkpoints(args.checkpoints)\n    run(sys.executable, "-c", "from melo.api import TTS; from openvoice.api import ToneColorConverter; print(\'Model imports OK\')")\n    marker = {"openvoice": OPENVOICE_REV, "melo": MELO_REV, "checkpoints": CHECKPOINT_REV,\n              "torch": "2.5.1", "index": args.torch_index}\n    (Path(sys.prefix) / "manhua-ready.json").write_text(json.dumps(marker), encoding="utf-8")\n    print("Runtime installed. Model weights/tokenizers may download on first inference.")\n\n\nif __name__ == "__main__":\n    main()\n', 'prepare_voice.py': '"""Prepare private OpenVoice V2 assets in a notebook session; no web server."""\nimport argparse\nimport json\nimport tempfile\nimport zipfile\nfrom pathlib import Path\n\nfrom voice_assets import cache_name, normalize_text, validate_wav\nfrom setup_runtime import OPENVOICE_REV, MELO_REV, CHECKPOINT_REV\n\nTEST_TEXT = "안녕하세요. 오늘도 한국어 공부를 시작해 볼까요?"\n\n\ndef prepare(reference, checkpoints, output, phrases, device=None):\n    import numpy as np\n    import soundfile as sf\n    import torch\n    from openvoice.api import ToneColorConverter\n    from engine import KoreanVoice\n\n    reference, checkpoints, output = Path(reference), Path(checkpoints), Path(output)\n    texts = list(dict.fromkeys(normalize_text(t) for t in [TEST_TEXT, *phrases]))\n    if len(texts) > 10000:\n        raise ValueError("At most 10,000 phrases per bundle")\n    device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")\n    info = sf.info(reference)\n    if info.duration < 3 or info.duration > 600:\n        raise ValueError("Use a clean single-speaker WAV lasting 3–600 seconds")\n    audio, sr = sf.read(reference, dtype="float32", always_2d=True)\n    audio = audio.mean(axis=1)\n    if not np.isfinite(audio).all() or np.max(np.abs(audio)) < 0.001:\n        raise ValueError("Reference is silent or invalid")\n    # A prepared, clean recording needs no Whisper, VAD model or ffmpeg.\n    # Short voiced chunks bound extraction memory; silence-only chunks are skipped.\n    with tempfile.TemporaryDirectory(prefix="mhl-private-") as tmp:\n        tmp = Path(tmp)\n        chunks = []\n        for offset in range(0, len(audio), 10 * sr):\n            chunk = audio[offset:offset + 10 * sr]\n            if len(chunk) < sr or float(np.sqrt(np.mean(chunk ** 2))) < 0.003:\n                continue\n            path = tmp / f"reference-{len(chunks)}.wav"\n            sf.write(path, chunk, sr, subtype="PCM_16")\n            chunks.append(str(path))\n        if not chunks:\n            raise ValueError("No usable speech chunks; trim silence and check the recording level")\n        converter = ToneColorConverter(str(checkpoints / "converter/config.json"), device=device)\n        converter.load_ckpt(str(checkpoints / "converter/checkpoint.pth"))\n        embedding = tmp / "target_se.pth"\n        converter.extract_se(chunks, se_save_path=str(embedding))\n        del converter\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n        voice = KoreanVoice(checkpoints, embedding, device)\n        cache = tmp / "cache"\n        cache.mkdir()\n        for index, text in enumerate(texts):\n            (cache / cache_name(text)).write_bytes(validate_wav(voice.synthesize(text)))\n            print(f"Prepared {index + 1}/{len(texts)} phrases")\n        manifest = {"format": "manhua-lens-private-v1", "private": True,\n                    "language": "KR", "openvoice": OPENVOICE_REV, "melo": MELO_REV,\n                    "checkpoints": CHECKPOINT_REV,\n                    "phrases": len(texts), "normalization": "NFC+strip", "speed": 0.95}\n        output.parent.mkdir(parents=True, exist_ok=True)\n        # Explicit allowlist: raw recordings and intermediate chunks never exported.\n        with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as bundle:\n            bundle.writestr("manifest.json", json.dumps(manifest))\n            bundle.write(embedding, "target_se.pth")\n            for path in sorted(cache.glob("*.wav")):\n                bundle.write(path, "cache/" + path.name)\n    print("PRIVATE bundle ready. Download it before ending the session; do not publish it.")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--reference", type=Path, required=True)\n    parser.add_argument("--checkpoints", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--phrases", type=Path, help="UTF-8 text file, one phrase per line")\n    parser.add_argument("--device", choices=["cpu", "cuda:0"])\n    args = parser.parse_args()\n    phrases = args.phrases.read_text(encoding="utf-8").splitlines() if args.phrases else []\n    prepare(args.reference, args.checkpoints, args.output, [p for p in phrases if p.strip()], args.device)\n', 'requirements.txt': '# HTTP layer only. Full model setup: python setup_runtime.py\nfastapi==0.115.6\nuvicorn==0.34.0\n', 'requirements-models.txt': "# Python 3.10; installed by setup_runtime.py into an isolated environment.\n# OpenVoice's numpy==1.22 and ASR dependencies are intentionally bypassed.\n# Melo imports multilingual cleaners eagerly, so their text dependencies remain.\nnumpy==1.26.4\ntorch==2.5.1\ntorchaudio==2.5.1\nlibrosa==0.9.1\nnumba==0.60.0\nsoundfile==0.12.1\ntransformers==4.27.4\nhuggingface-hub==0.25.2\ncached_path==1.6.7\ntxtsplit==1.0.0\nnum2words==0.5.12\nunidic_lite==1.0.8\nunidic==1.1.0\nmecab-python3==1.0.9\npykakasi==2.2.1\nfugashi==1.3.0\ng2p_en==2.1.0\nanyascii==0.3.2\njamo==0.4.1\ngruut[de,es,fr]==2.2.3\ng2pkk==0.1.2\npython-mecab-ko==1.3.7\npydub==0.25.1\neng_to_ipa==0.0.2\ninflect==7.0.0\nunidecode==1.3.7\npypinyin==0.50.0\ncn2an==0.5.22\njieba==0.42.1\nlangid==1.1.6\ntqdm==4.67.1\ntensorboard==2.16.2\nloguru==0.7.2\nnltk==3.8.1\nwavmark==0.0.3\n"}
for name, source in PUBLIC_SOURCES.items():
    (SCRIPTS / name).write_text(source, encoding='utf-8')
SESSION = Path('/tmp/manhua-private-session')
SESSION.mkdir(mode=0o700, exist_ok=True)
TOOLS = Path('/tmp/manhua-uv')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(TOOLS), 'uv==0.6.17'], check=True)
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(TOOLS)
VENV = Path('/tmp/manhua-python310')
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.10', '--seed', str(VENV)], env=ENV, check=True)
PYTHON = str(VENV / 'bin/python')
CHECKPOINTS = Path('/tmp/manhua-checkpoints-v2')
# cu121 supports the pinned PyTorch build; choose cpu if no free GPU is available.
TORCH_INDEX = 'cu121'
subprocess.run([PYTHON, str(SCRIPTS / 'setup_runtime.py'), '--torch-index', TORCH_INDEX,
                '--checkpoints', str(CHECKPOINTS)], check=True)


## Upload the consented reference into this session only
Choose `openvoice_reference_korean_full.wav`. Use clean single-speaker speech,
without music or other voices (3–600 seconds). This workflow averages short
voiced chunks; it does not use an ASR model to clean a noisy recording.
If Kaggle does not render the widget, use this notebook in free Colab and the
Colab upload fallback below. Do not substitute a public or private Dataset upload.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
upload = widgets.FileUpload(accept='.wav', multiple=False)
display(upload)


In [ ]:
REFERENCE = SESSION / 'openvoice_reference_korean_full.wav'
if upload.value:
    # Support ipywidgets 7 (dict) and 8 (tuple).
    items = list(upload.value.values()) if isinstance(upload.value, dict) else list(upload.value)
    assert len(items) == 1
    item = items[0]
    assert item.get('name', item.get('metadata', {}).get('name')) == REFERENCE.name
    assert len(item['content']) <= 128 * 1024 * 1024, 'Reference must be under 128 MiB'
    REFERENCE.write_bytes(bytes(item['content']))
    upload.value = {} if isinstance(upload.value, dict) else ()
    upload.close()
    del items, item
else:
    assert REFERENCE.is_file(), 'Upload the WAV above, or use the Colab fallback cell.'
print('Private reference is in session temporary storage.')


In [ ]:
# OPTIONAL Colab fallback: run only if the widget upload was unavailable.
# from google.colab import files
# previous = Path.cwd()
# os.chdir(SESSION)
# try:
#     uploaded = files.upload()
#     assert list(uploaded) == ['openvoice_reference_korean_full.wav']
#     del uploaded
# finally:
#     os.chdir(previous)
# REFERENCE = SESSION / 'openvoice_reference_korean_full.wav'


In [ ]:
# Add exact words AND sentences you want available in model-free local mode.
# Punctuation and internal spaces matter. The test sentence is always included.
PHRASES = [
    '안녕하세요.',
    '한국어',
    '공부',
    '오늘도 한국어 공부를 시작해 볼까요?',
]
phrases_file = SESSION / 'phrases.txt'
phrases_file.write_text('\n'.join(PHRASES), encoding='utf-8')
BUNDLE = SESSION / 'manhua-private.zip'
subprocess.run([PYTHON, str(SCRIPTS / 'prepare_voice.py'), '--reference', str(REFERENCE),
                '--checkpoints', str(CHECKPOINTS), '--output', str(BUNDLE),
                '--phrases', str(phrases_file)], check=True)


In [ ]:
# Private listening test. Clear this output before saving the notebook.
import zipfile
import hashlib
import unicodedata
from IPython.display import Audio
test_text = '안녕하세요. 오늘도 한국어 공부를 시작해 볼까요?'
key = hashlib.sha256(unicodedata.normalize('NFC', test_text).strip().encode('utf-8')).hexdigest()
with zipfile.ZipFile(BUNDLE) as archive:
    display(Audio(archive.read('cache/' + key + '.wav')))


In [ ]:
# Download without copying private assets into Kaggle's saved output directory.
# The Kaggle link contains private bytes: clear ALL outputs after downloading.
import base64
from IPython.display import HTML
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(BUNDLE))
else:
    encoded = base64.b64encode(BUNDLE.read_bytes()).decode('ascii')
    display(HTML('<a download="manhua-private.zip" href="data:application/zip;base64,'
                 + encoded + '">Download PRIVATE local assets</a>'))
    del encoded
# If Kaggle/browser blocks the download link, rerun this notebook in free Colab
# and use its native download above. Do not publish assets as a workaround.


## Local Windows use
With free Python 3 installed, run from the Manhua Lens repository:

```powershell
.\voice_server\start.cmd -Bundle "C:\Users\YOU\Downloads\manhua-private.zip"
```

Later: `.\voice_server\start.cmd`. The endpoint is still
`http://127.0.0.1:8765/tts`; use Manhua Lens normally. Cache misses immediately
fall back to installed Windows/browser Korean TTS. There is no WSL/PyTorch
requirement for cache mode. For arbitrary-text cloning, optional
`.\voice_server\start.cmd -Mode live` installs a substantial CPU model runtime.
Only import bundles you produced yourself. The ZIP exports `target_se.pth`,
cloned phrase WAVs, and a version manifest; it never exports the reference WAV.

After downloading and listening: clear all notebook outputs, delete the upload
widget, and stop/delete the session. Never publish a notebook with private outputs.


In [ ]:
# Optional cleanup AFTER the private ZIP has been downloaded successfully.
# Only the fixed private session directory is removed.
import shutil
from IPython.display import clear_output
assert SESSION == Path('/tmp/manhua-private-session')
shutil.rmtree(SESSION)
clear_output()
# Use the notebook UI's Clear ALL Outputs too (this only clears this cell).
